[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/17_dropout.ipynb)

# 🟢 简单：实现 Dropout

从零实现 **Dropout** 正则化。

### 函数签名
```python
class MyDropout(nn.Module):
    def __init__(self, p: float = 0.5): ...
    def forward(self, x: Tensor) -> Tensor: ...
```

### 规则
- **训练阶段**：以概率 `p` 将每个元素置零，并将剩余元素缩放 `1/(1-p)` 倍
- **评估阶段**：直接返回输入（恒等映射）
- 禁止使用 `nn.Dropout` 或 `F.dropout`

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn

In [ ]:
# ✏️ 在此实现你的代码

class MyDropout(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        pass

    def forward(self, x):
        pass

1. 为什么要除以 $(1-p)$？如果在训练阶段只是简单地将部分神经元置零，那么在推理阶段（所有神经元都工作时），输出的数值总和会比训练阶段大。不缩放的情况：推理时需要手动将权重乘以 $(1-p)$。反向缩放（本代码采用）：在训练时直接除以 $(1-p)$，这样推理时就无需任何修改，直接作为恒等映射即可。
2. self.training 的重要性nn.Module 自带 training 布尔属性。当你调用 model.eval() 时，该属性变为 False，确保了模型在测试时不会随机丢弃信息。
3. 概率边界当 $p=0$ 时，Dropout 实际上变成了恒等变换；注意代码逻辑中应避免 $p=1$ 导致除以零的错误（通常 Dropout 概率不会设为 1）。

In [ ]:
class MyDropout(nn.Module):
    def __init__(self, p: float = 0.5):
        """
        Args:
            p: 元素被置零的概率。
        """
        super().__init__()
        # 边界检查
        if not 0 <= p < 1:
            raise ValueError(f"Dropout probability must be in [0, 1), got {p}")
        self.p = p

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 1. 评估阶段 (inference)：直接返回输入
        if not self.training or self.p == 0:
            return x

        # 2. 训练阶段 (training)
        # 生成与输入形状相同的随机噪声，取值范围在 [0, 1)
        # 如果随机值大于 p，则保留（设为1），否则舍弃（设为0）
        # 这里的 mask 形状与 x 一致
        mask = (torch.rand(x.shape, device=x.device) > self.p).float()

        # 3. 反向缩放 (Inverted Dropout)
        # 为了保持训练和评估时神经元输出的期望值不变：
        # E = p * 0 + (1 - p) * (x / (1 - p)) = x
        return x * mask / (1 - self.p)

In [ ]:
# 🧪 调试
d = MyDropout(p=0.5)
d.train()
x = torch.ones(10)
print('训练模式:', d(x))
d.eval()
print('评估模式:', d(x))

In [ ]:
# ✅ 提交
from torch_judge import check
check('dropout')